[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/04_resnet50.ipynb)

In [ ]:
import os, sys
from pathlib import Path

# ── Detectar si estamos en Google Colab ──────────────────────────────────
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

DATA_ROOT_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Estrategia 1: acceso por ID de carpeta raíz del proyecto
    _p1 = "/content/drive/.shortcut-targets-by-id/13VEeA8Jt0G_NNpJelWZrhacBhA_mRpnU"
    # Estrategia 2: acceso por ID de TrainData (subir un nivel)
    _p2 = "/content/drive/.shortcut-targets-by-id/13Dj5DYGOwgMGomMC8gCySKGr0ne8hOpU"

    if os.path.exists(_p1) and os.path.isdir(_p1):
        DATA_ROOT_OVERRIDE = _p1
        print(f"Estrategia 1 OK: {_p1}")
    elif os.path.exists(_p2) and os.path.isdir(_p2):
        DATA_ROOT_OVERRIDE = str(Path(_p2).parent)
        print(f"Estrategia 2 OK: {DATA_ROOT_OVERRIDE}")
    else:
        import subprocess
        _r = subprocess.run(
            ["find", "/content/drive/MyDrive", "-name", "TrainData",
             "-type", "d", "-maxdepth", "6"],
            capture_output=True, text=True, timeout=30
        )
        _hits = [p.replace('/TrainData', '') for p in _r.stdout.strip().splitlines() if p]
        if _hits:
            DATA_ROOT_OVERRIDE = _hits[0]
            print(f"Estrategia 3 OK: {DATA_ROOT_OVERRIDE}")
        else:
            print("No se encontró el dataset. Ajusta DATA_ROOT_OVERRIDE manualmente:")
            print("  DATA_ROOT_OVERRIDE = 'ruta/en/tu/Drive'")

    if DATA_ROOT_OVERRIDE:
        print(f"Dataset: {DATA_ROOT_OVERRIDE}")
else:
    print('Entorno local — se usará detección automática de rutas.')


# 04 — Fine-tuning ResNet-50

Protocolo de fine-tuning en dos fases:
1. **Fase 1** (5 épocas): backbone congelado, solo se entrena la cabeza de clasificación
2. **Fase 2** (épocas restantes): backbone descongelado, fine-tuning completo con LR diferencial

---

In [ ]:
import os, sys
from pathlib import Path

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN DEL DATASET
# Si la celda de Drive de arriba encontró el dataset, DATA_ROOT_OVERRIDE
# ya está definida. Si no, puedes ajustarla manualmente aquí:
# DATA_ROOT_OVERRIDE = "/content/drive/MyDrive/landslide4sense"
# ══════════════════════════════════════════════════════════════════════════════

# Preservar DATA_ROOT_OVERRIDE si ya fue fijada por la celda de Drive
try:
    DATA_ROOT_OVERRIDE
except NameError:
    DATA_ROOT_OVERRIDE = None

# ── Detección automática ──────────────────────────────────────────────────────
if DATA_ROOT_OVERRIDE:
    DATA_ROOT = Path(DATA_ROOT_OVERRIDE)
    print(f"Usando ruta configurada: {DATA_ROOT}")
else:
    _cwd = Path(os.getcwd())
    _candidates = [
        _cwd / ".." / "data",
        _cwd / "..",
        _cwd / "data",
        _cwd,
        Path("/content/drive/MyDrive/landslide4sense"),
        Path("/content/drive/MyDrive/Landslide_ML"),
        Path("/content/landslide4sense"),
        Path("/content/data"),
        Path("/content"),
    ]
    DATA_ROOT = None
    for _c in _candidates:
        if (_c / "TrainData" / "img").exists():
            DATA_ROOT = _c.resolve()
            print(f"Dataset detectado automáticamente en: {DATA_ROOT}")
            break

if DATA_ROOT is None or not (DATA_ROOT / "TrainData" / "img").exists():
    print("""
⚠️  Dataset no encontrado. Opciones:

━━━  OPCIÓN A — Google Drive (recomendado)  ━━━
  Ejecuta la celda de Drive de arriba y vuelve a correr esta celda.

━━━  OPCIÓN B — Kaggle API  ━━━
  !pip install kaggle -q
  # Sube tu kaggle.json y ejecuta:
  !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
  !kaggle datasets download -d landslide4sense/competition -p /content/
  !unzip -q /content/competition.zip -d /content/landslide4sense/
  DATA_ROOT_OVERRIDE = "/content/landslide4sense"
""")
    raise FileNotFoundError("Dataset no encontrado. Establece DATA_ROOT_OVERRIDE con la ruta correcta.")

# ── Verificar estructura del dataset ─────────────────────────────────────────
n_train = len(list((DATA_ROOT / "TrainData" / "img").glob("*.h5")))
print(f"✓ Dataset encontrado en: {DATA_ROOT}")
print(f"  TrainData/img : {n_train} archivos .h5")


## 1. Verificar la adaptación de la primera capa conv

In [ ]:
from src.models import build_resnet50, model_summary
import torch

model = build_resnet50(n_channels=14, pretrained=True)
print(model_summary(model, n_channels=14))

# Verificar forma de la primera capa
first_conv = model.features[0]
print(f"\nPrimera conv adaptada:")
print(f"  in_channels:  {first_conv.in_channels}")
print(f"  out_channels: {first_conv.out_channels}")
print(f"  weight shape: {first_conv.weight.shape}  ← (out_c, 14, 7, 7)")

# Test forward pass
dummy = torch.randn(2, 14, 128, 128)
out   = model(dummy)
print(f"\nForward pass OK: (2,14,128,128) → {out.shape}")

## 2. Protocolo de fine-tuning — Fase 1: congelar backbone

In [ ]:
# Congelar backbone y contar parámetros entrenables
model.freeze_backbone()
print("Backbone congelado.")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Parámetros entrenables: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)")
print("→ Solo la cabeza de clasificación se entrena en las primeras 5 épocas.")

## 3. Ejecutar K-Fold Cross-Validation completo

In [ ]:
# ⚠️ Este paso puede tomar 2-4 horas en GPU. Reducir epochs para prueba rápida.
# cfg.epochs = 5   # Descomenta para prueba rápida

summary = run_kfold(cfg)
print("\nEntrenamiento completado.")

## 4. Visualizar historia de entrenamiento

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

output_dir = Path("../results/resnet50")

# Cargar y visualizar el mejor fold
summary_path = output_dir / "kfold_summary.json"
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    
    print(f"\nResumen 5-Fold CV — ResNet-50 Fine-tuned")
    print(f"{'─'*50}")
    for k, v in summary["aggregated"].items():
        print(f"  {k:20s}: {v['mean']:.4f} ± {v['std']:.4f}")
else:
    print("Ejecutar la celda de entrenamiento primero.")

## 5. Evaluación con umbral óptimo

In [ ]:
from src.evaluate import evaluate_model, find_optimal_threshold
from src.dataset import Landslide4SenseDataset, get_fold_indices, get_dataloaders
from src.models import build_resnet50
from src.utils import load_checkpoint, get_device
import torch

device = get_device()
fold_idx = 0   # Usar el primer fold como ejemplo

# Cargar el mejor modelo del fold 0
ckpt_path = Path(f"../results/resnet50/fold_{fold_idx}/best_model.pth")
if ckpt_path.exists():
    full_ds = Landslide4SenseDataset(
        img_dir="../data/TrainData/img", mask_dir="../data/TrainData/mask", normalize=False
    )
    folds = get_fold_indices(full_ds, n_folds=5, seed=42)
    _, val_idx = folds[fold_idx]
    loaders = get_dataloaders(cfg, train_indices=None, val_indices=val_idx)
    
    model = build_resnet50(n_channels=14, pretrained=False).to(device)
    load_checkpoint(model, str(ckpt_path), device=device)
    
    metrics = evaluate_model(
        model, loaders["val"], device,
        optimize_threshold=True,
        model_name="ResNet-50 Fine-tuned",
        output_dir="../results/resnet50/evaluation",
    )
else:
    print("Checkpoint no encontrado. Ejecutar entrenamiento primero.")